In [1]:
! pip3 install transformer_lens einops eindex-callum jaxtyping git+https://github.com/callummcdougall/CircuitsVis.git#subdirectory=python

  Cloning https://github.com/callummcdougall/CircuitsVis.git to /private/var/folders/x0/x0hrrnl51dn8nbwxccj6p8d00000gp/T/pip-req-build-6wd12wjn
  Running command git clone --filter=blob:none --quiet https://github.com/callummcdougall/CircuitsVis.git /private/var/folders/x0/x0hrrnl51dn8nbwxccj6p8d00000gp/T/pip-req-build-6wd12wjn
  Resolved https://github.com/callummcdougall/CircuitsVis.git to commit 1e6129d08cae7af9242d9ab5d3ed322dd44b4dd3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import functools
import sys
from pathlib import Path
from typing import Callable, List
from dataclasses import dataclass, field

import circuitsvis as cv
import einops
import numpy as np

import torch as t, torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau

from eindex import eindex
from IPython.display import display
from jaxtyping import Float, Int
from torch import Tensor
from tqdm import tqdm
from transformer_lens import (
    ActivationCache,
    FactoredMatrix,
    HookedTransformer,
    HookedTransformerConfig,
    utils,
)
from transformer_lens.hook_points import HookPoint

In [3]:
import torch as t
device = t.device(
    "cuda" if t.cuda.is_available() else "cpu"
)
device

device(type='cpu')

In [4]:
from transformer_lens import HookedTransformer
gpt2_small: HookedTransformer = HookedTransformer.from_pretrained("gpt2-small")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer


In [5]:
# Generating a cache from a forward pass to play around with.
gpt2_text = "Natural language processing tasks, such as question answering, machine translation, reading comprehension, and summarization, are typically approached with supervised learning on task-specific datasets."
gpt2_tokens = gpt2_small.to_tokens(gpt2_text)
gpt2_logits, gpt2_cache = gpt2_small.run_with_cache(gpt2_tokens, remove_batch_dim=True)

In [6]:
# List of activations that can be accessed.
for key in gpt2_cache.keys():
    print(key)

hook_embed
hook_pos_embed
blocks.0.hook_resid_pre
blocks.0.ln1.hook_scale
blocks.0.ln1.hook_normalized
blocks.0.attn.hook_q
blocks.0.attn.hook_k
blocks.0.attn.hook_v
blocks.0.attn.hook_attn_scores
blocks.0.attn.hook_pattern
blocks.0.attn.hook_z
blocks.0.hook_attn_out
blocks.0.hook_resid_mid
blocks.0.ln2.hook_scale
blocks.0.ln2.hook_normalized
blocks.0.mlp.hook_pre
blocks.0.mlp.hook_post
blocks.0.hook_mlp_out
blocks.0.hook_resid_post
blocks.1.hook_resid_pre
blocks.1.ln1.hook_scale
blocks.1.ln1.hook_normalized
blocks.1.attn.hook_q
blocks.1.attn.hook_k
blocks.1.attn.hook_v
blocks.1.attn.hook_attn_scores
blocks.1.attn.hook_pattern
blocks.1.attn.hook_z
blocks.1.hook_attn_out
blocks.1.hook_resid_mid
blocks.1.ln2.hook_scale
blocks.1.ln2.hook_normalized
blocks.1.mlp.hook_pre
blocks.1.mlp.hook_post
blocks.1.hook_mlp_out
blocks.1.hook_resid_post
blocks.2.hook_resid_pre
blocks.2.ln1.hook_scale
blocks.2.ln1.hook_normalized
blocks.2.attn.hook_q
blocks.2.attn.hook_k
blocks.2.attn.hook_v
blocks.2.att

Diagram demonstrating the flow of information in each transformer block:        
https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/full-merm.svg

Activation shorthands for the cache:    
residual stream (pre): "resid_pre", layer         
layer norm 1 (scaling factor): "scale", layer, "ln1"        
layer norm 1 (normalized): "normalized", layer, "ln1"            
attention q: "q", layer       
attention k: "k", layer      
attention v: "v", layer     
attention scores: "attn_scores", layer      
attention pattern: "pattern", layer       
z: "z", layer           
output from attention: "attn_out", layer         
residual stream (mid): "resid_mid", layer         
layer norm 2 (scaling factor): "scale", layer, "ln2"        
layer norm 2 (normalized): "normalized", layer, "ln2"    
mlp pre: "pre", layer              
mlp post: "post", layer           
mlp outputs: "mlp_out", layer         
residual stream (post): "resid_post", layer           

In [7]:
layer0_pattern_from_cache = gpt2_cache["pattern", 0]
mlp_in = gpt2_cache["pre", 0]
mlp_out = gpt2_cache["post", 0]
mlp_final = gpt2_cache["mlp_out", 0]

print("Attention pattern things")
print(layer0_pattern_from_cache.shape)
print(gpt2_tokens.shape)

print("\nmlp layer things")
print(mlp_in.shape)
print(mlp_out.shape)
print(mlp_final.shape)

Attention pattern things
torch.Size([12, 33, 33])
torch.Size([1, 33])

mlp layer things
torch.Size([33, 3072])
torch.Size([33, 3072])
torch.Size([33, 768])


In [8]:
# getting model configuration info
config = gpt2_small.cfg
config

HookedTransformerConfig:
{'NTK_by_parts_factor': 8.0,
 'NTK_by_parts_high_freq_factor': 4.0,
 'NTK_by_parts_low_freq_factor': 1.0,
 'NTK_original_ctx_len': 8192,
 'act_fn': 'gelu_new',
 'attention_dir': 'causal',
 'attn_only': False,
 'attn_scale': 8.0,
 'attn_scores_soft_cap': -1.0,
 'attn_types': None,
 'checkpoint_index': None,
 'checkpoint_label_type': None,
 'checkpoint_value': None,
 'd_head': 64,
 'd_mlp': 3072,
 'd_model': 768,
 'd_vocab': 50257,
 'd_vocab_out': 50257,
 'decoder_start_token_id': None,
 'default_prepend_bos': True,
 'device': device(type='mps'),
 'dtype': torch.float32,
 'eps': 1e-05,
 'experts_per_token': None,
 'final_rms': False,
 'from_checkpoint': False,
 'gated_mlp': False,
 'init_mode': 'gpt2',
 'init_weights': False,
 'initializer_range': 0.02886751345948129,
 'load_in_4bit': False,
 'model_name': 'gpt2',
 'n_ctx': 1024,
 'n_devices': 1,
 'n_heads': 12,
 'n_key_value_heads': None,
 'n_layers': 12,
 'n_params': 84934656,
 'normalization_type': 'LNPre',
 '

Code below here mostly taken from Open CLT:          
https://github.com/etredal/openCLT

In [9]:
from typing import Tuple, Any

# JumpReLU implementation by OpenCLT
def rectangle(x: torch.Tensor) -> torch.Tensor:
    return ((x > -0.5) & (x < 0.5)).to(x)

class jumprelu(torch.autograd.Function):
    @staticmethod
    def forward(x: torch.Tensor, threshold: torch.Tensor, bandwidth: float) -> torch.Tensor:
        return (x * (x > threshold)).to(x)

    @staticmethod
    def setup_context(
        ctx: Any, inputs: Tuple[torch.Tensor, torch.Tensor, float], output: torch.Tensor
        ) -> None:
        x, threshold, bandwidth = inputs
        del output
        ctx.save_for_backward(x, threshold)
        ctx.bandwidth = bandwidth

    @staticmethod
    def backward(ctx: Any, grad_output: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, None]:
        x, threshold = ctx.saved_tensors
        bandwidth = ctx.bandwidth
        # Gradient is 1 where x > threshold, scaled by bandwidth
        x_grad = (x >threshold) * grad_output
        threshold_grad = torch.sum(
            -(threshold / bandwidth) * rectangle((x - threshold) / bandwidth) * grad_output,
            dim=0
        )
        return x_grad, threshold_grad, None

class JumpReLU(t.nn.Module):
    def __init__(self, threshold: float, bandwidth: float) -> None:
        super().__init__()
        self.threshold = nn.Parameter(t.tensor(threshold))
        self.bandwidth = bandwidth

    def forward(self, x: t.Tensor) -> t.Tensor:
        return jumprelu.apply(x, self.threshold, self.bandwidth)
    
    def extra_repr(self) -> str:
        return f"threshold={self.threshold}, bandwidth={self.bandwidth}"
    
# TrainingMetric class implementation by OpenCLT
@dataclass
class TrainingMetric:
    total_loss:             list[float] = field(default_factory=list)
    reconstruction_loss:    list[float] = field(default_factory=list)
    sparsity_loss:          list[float] = field(default_factory=list)
    l0_metric:              list[float] = field(default_factory=list)
    learning_rate:          list[float] = field(default_factory=list) 

In [10]:
class CLT(nn.Module):
    def __init__(self, n_features):
        super().__init__()

        # Load the model
        self.model: HookedTransformer = HookedTransformer.from_pretrained("gpt2-small")
        self.config = self.model.cfg
        
        # Store important values
        self.d_model = config.d_model
        self.num_layers = config.n_layers
        self.num_features = n_features
        
        self.device = t.device(
            "mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu"
        )

        # Freeze base model parameters
        for param in self.model.parameters():
            param.requires_grad = False

        # Create encoder for each layer
        self.encoders = nn.ModuleList([
            nn.Linear(self.d_model, self.num_features) 
            for _ in range(self.num_layers)
        ]).to(self.device)

        # Create decoder for each layer
        self.decoders = nn.ModuleDict()
        for encoder_layer in range (self.num_layers):
            for decoder_layer in range (encoder_layer, self.num_layers): # current layer and all subsequent layers only (no backwards)
                key = f"layer{encoder_layer}_decoder{decoder_layer}"
                self.decoders[key] = nn.Linear(self.num_features, self.d_model).to(self.device)

        # initializing CLT weights
        for encoder_layer in range(self.num_layers):
            # Initialize encoder weights according to the anthropic paper
            std_encoder = 1.0 / np.sqrt(self.num_features)
            nn.init.uniform_(self.encoders[encoder_layer].weight, -std_encoder, std_encoder)
            nn.init.zeros_(self.encoders[encoder_layer].bias)
            
            # Initialize all decoders for this encoder layer according to the anthropic paper
            for decoder_layer in range(encoder_layer, self.num_layers):
                key = f"layer{encoder_layer}_decoder{decoder_layer}"
                # Initialize decoder weights
                std_decoder = 1.0 / np.sqrt(self.num_layers * self.d_model)  
                nn.init.uniform_(self.decoders[key].weight, -std_decoder, std_decoder)
                nn.init.zeros_(self.decoders[key].bias)

        # initializing activation functions
        self.activation_functions = nn.ModuleList([
            JumpReLU(threshold=0.03, bandwidth=1.0)  # Example threshold and bandwidth from the paper
                for _ in range(self.num_layers)
            ]).to(self.device)
        
        self.hooks = []
        self.mlp_pre = {}
        self.mlp_post = {}
        self.register_hooks()

    # creating hook functions and storing in self.hooks
    def register_hooks(self):
        def hook_pre_mlp(residual: Float[Tensor, "seq d_model"], hook: HookPoint, layer_idx: int) -> Float[Tensor, "seq d_model"]:
            self.mlp_pre[f"layer_{layer_idx}"] = residual
            return residual
        
        def hook_post_mlp(residual: Float[Tensor, "seq d_model"], hook: HookPoint, layer_idx: int) -> Float[Tensor, "seq d_model"]:
            self.mlp_post[f"layer_{layer_idx}"] = residual
            return residual
        
        for layer_idx in range(self.num_layers):
            temp_hook_pre = functools.partial(hook_pre_mlp, layer_idx=layer_idx)
            temp_hook_post = functools.partial(hook_post_mlp, layer_idx=layer_idx)

            # TODO figure out which activations in the transformers should be read out

            self.hooks.append((utils.get_act_name('normalized', layer_idx), temp_hook_pre))
            self.hooks.append((utils.get_act_name('mlp_out', layer_idx), temp_hook_post))

            # TODO figure out which activations in the transformers should be read out

    
    def train_clt(self,
                  texts: List[str], 
                  batch_size: int = 4, 
                  num_epochs: int = 5,
                  learning_rate: float = 1e-4,
                  l1_sparsity_coefficient: float = 0.05,
                  lr_scheduler_factor: float = 0.1,
                  lr_scheduler_patience: int = 100,
                 ) -> TrainingMetric:
        
        # Set model to training mode
        self.train()
        
        # Gather all of the trainable parameters
        all_params = (list(self.encoders.parameters()) 
                      + list(self.decoders.parameters()) 
                      + list(self.activation_functions.parameters()))

        # Create optimizer for all of the trainable parameters
        optimizer = torch.optim.Adam(
            all_params, lr=learning_rate
        )

        scheduler = ReduceLROnPlateau(optimizer,
                                      mode='min', # We want the loss to decrease
                                      factor=lr_scheduler_factor,
                                      patience=lr_scheduler_patience
        )        
        # Training metrics
        metrics = TrainingMetric()
        
        # Tokenize all texts
        encoded_texts = [self.model.tokenizer.encode(text, return_tensors="pt").to(self.device) for text in texts]

        for epoch in range(num_epochs):
            epoch_loss = 0
            epoch_recon_loss = 0
            epoch_sparsity_loss = 0
            epoch_L0 = 0

            # Batch data
            num_batches = (len(encoded_texts) + batch_size - 1) // batch_size

            for batch_idx in tqdm(range(num_batches), desc=f"Epoch {epoch+1}/{num_epochs}"):
                
                # Get current batch of data
                start_idx = batch_idx * batch_size
                end_idx = min(start_idx + batch_size, len(encoded_texts))
                batch_texts = encoded_texts[start_idx:end_idx]
                
                # Batch preprocessing - pad to same length within batch
                max_len = max(text.size(1) for text in batch_texts)
                padded_texts = []
                attention_masks = []
                
                for text in batch_texts:
                    pad_len = max_len - text.size(1)
                    padded_text = F.pad(text, (0, pad_len), value=self.tokenizer.pad_token_id)
                    mask = torch.ones_like(padded_text, dtype=torch.float)
                    mask[:, -pad_len:] = 0 if pad_len > 0 else 1
                    
                    padded_texts.append(padded_text)
                    attention_masks.append(mask)

                # End of batch preprocessing
                input_ids = torch.cat(padded_texts, dim=0)
                attention_mask = torch.cat(attention_masks, dim=0)

                # Zero out gradients - avoid gradients accumulating between batches
                optimizer.zero_grad()

                self.mlp_pre.clear()
                self.mlp_post.clear()

                # Generate MLP activations
                self.model.run_with_hooks(input_ids=input_ids, attention_mask=attention_mask, fwd_hooks=self.hooks, reset_hooks_end=False)

                # Calculate loss for each layer
                total_loss = 0
                reconstruction_loss = 0
                sparsity_loss = 0

                # Feature store for reconstruction loss
                all_features = {}

                for layer_idx in range(self.num_layers):
                    if f"layer_{layer_idx}" in self.mlp_pre:
                        pre = self.mlp_pre[f"layer_{layer_idx}"]
                        post = self.mlp_post[f"layer_{layer_idx}"]

                        # TODO figure out why we are normalizing here

                        pre_normalized = F.layer_norm(pre, pre.shape[-1:])  # Normalize input to MLP
                        post_normalized = F.layer_norm(post, post.shape[-1:])  # Normalize output of MLP

                        # TODO figure out why we are normalizing here

                        # Compute feature activations based on MLP inputs and store for future decoding
                        features = self.encoders[layer_idx](pre_normalized)
                        feature_activations = self.activation_functions[layer_idx](features)
                        all_features[f"features_{layer_idx}"] = feature_activations

                        # Compute reconstructed output using contributions from all previous layers (and current layer!)
                        post_reconstructed = t.zeros_like(post)
                        for encoder_layer in range(layer_idx + 1):                                                      # For every prior layer until the current layer
                            if f"features_{encoder_layer}" in all_features:                                             # If we have stored encoded features for the layer
                                key = f"layer{encoder_layer}_decoder{layer_idx}"                                        # Decode the features with the decoder for this layer
                                post_reconstructed += self.decoders[key](all_features[f"features_{encoder_layer}"])     # Add the decoded features to our reconstruction

                        # === Loss Calculations ===

                        # L0 metric (sparsity)
                        L0_metric = t.mean((feature_activations > 1e-6).float())

                        # Reconstruction loss (MSE)
                        recon = F.mse_loss(post_reconstructed, post_normalized)
                        reconstruction_loss += recon

                        # Sparsity Loss (L1 Regularization)
                        decoder_norms = []
                        for decoder_layer in range(layer_idx, self.num_layers):
                            
                            # TODO understand the computation happening to get our l1_loss

                            key = f"{layer_idx}_{decoder_layer}"
                            decoder_norm = torch.norm(self.decoders[key].weight, dim=1)
                            decoder_norms.append(decoder_norm)
                            total_decoder_norm = torch.stack(decoder_norms, dim=0).sum(dim=0)

                            # TODO understand the computation happening to get our l1_loss

                            c = 1.0
                            feature_sparsity = torch.tanh(c * total_decoder_norm * feature_activations.abs())
                            l1_loss = l1_sparsity_coefficient * torch.mean(feature_sparsity)
                            sparsity_loss += l1_loss

                        # Update total loss
                        total_loss += recon + l1_loss

                # Perform backwards pass
                total_loss.backward()

                # Clip gradients to and step the optimizaer
                torch.nn.utils.clip_grad_norm_(all_params, max_norm=1.0)
                optimizer.step()

                # Update metrics
                epoch_loss += total_loss.item()
                epoch_recon_loss += reconstruction_loss.item()
                if isinstance(sparsity_loss, torch.Tensor):
                    epoch_sparsity_loss += sparsity_loss.item()
                else:
                    epoch_sparsity_loss += sparsity_loss
                epoch_L0 += L0_metric.item()
            
            # Record epoch metrics
            avg_total_loss = epoch_loss / num_batches
            avg_recon_loss = epoch_recon_loss / num_batches
            avg_sparsity_loss = epoch_sparsity_loss / num_batches
            avg_l0_metric = epoch_L0 / num_batches
            
            metrics.total_loss.append(avg_total_loss)
            metrics.reconstruction_loss.append(avg_recon_loss)
            metrics.sparsity_loss.append(avg_sparsity_loss)
            metrics.l0_metric.append(avg_l0_metric)
            metrics.learning_rate.append(optimizer.param_groups[0]['lr'])

            # Step the scheduler
            scheduler.step(avg_total_loss) # Step with the monitored metric
            
            print(f"Epoch {epoch+1}/{num_epochs}: "
                  f"Loss = {avg_total_loss:.4f}, "
                  f"Recon = {avg_recon_loss:.4f}, "
                  f"Sparsity = {avg_sparsity_loss:.4f}, "
                  f"L0 Metric = {avg_l0_metric:.4f}")

            thresholds = [act.threshold.item() for act in self.activation_functions[:3]]
            print(f"JumpReLU thresholds first 3 layers: {[f'{t:.3f}' for t in thresholds]}")

        return metrics


In [11]:
# printing all modules
clt = CLT(10_000)

print("All named submodules:")
for name, module in clt.named_modules():
    print(f"Name: {name}, Module: {module}")

Loaded pretrained model gpt2-small into HookedTransformer
All named submodules:
Name: , Module: CLT(
  (model): HookedTransformer(
    (embed): Embed()
    (hook_embed): HookPoint()
    (pos_embed): PosEmbed()
    (hook_pos_embed): HookPoint()
    (blocks): ModuleList(
      (0-11): 12 x TransformerBlock(
        (ln1): LayerNormPre(
          (hook_scale): HookPoint()
          (hook_normalized): HookPoint()
        )
        (ln2): LayerNormPre(
          (hook_scale): HookPoint()
          (hook_normalized): HookPoint()
        )
        (attn): Attention(
          (hook_k): HookPoint()
          (hook_q): HookPoint()
          (hook_v): HookPoint()
          (hook_z): HookPoint()
          (hook_attn_scores): HookPoint()
          (hook_pattern): HookPoint()
          (hook_result): HookPoint()
        )
        (mlp): MLP(
          (hook_pre): HookPoint()
          (hook_post): HookPoint()
        )
        (hook_attn_in): HookPoint()
        (hook_q_input): HookPoint()
        